# 항만 대회: LLM 한글 표기 증강 v2 (표기 더 다양하게)

- 대회 공지(talkboard 418501, 10/23): train 데이터를 이용한 LLM 증강은 허용. 생성 데이터와 생성 코드는 보관.
- 입력: train 선박 5,851척의 영문명 (`llm_aug_input.csv`), 서로 겹치지 않는 예시 묶음 3개 (`llm_fewshot_sets.csv`, 각 30쌍)
- 모델: `Qwen/Qwen2.5-7B-Instruct` (Apache 2.0), vLLM
- 지시문 3종 × (greedy 1 + 샘플 2) = 배마다 최대 9개 표기 → `llm_korean_aug_v2.csv` (긴 형식: vessel_id, en_name, prompt_id, kind, ko)

**런타임 → 런타임 유형 변경 → A100 (또는 L4)** 로 바꾼 뒤:
1. 첫 셀을 실행합니다. 설치가 끝나면 **런타임이 자동으로 재시작**됩니다 (세션 충돌 메시지가 떠도 정상).
2. 재시작 후 **두 번째 셀부터** 끝까지 실행합니다. A100 기준 약 10~15분.

In [ ]:
# 1) vLLM 설치 후 자동 재시작 (설치가 PyTorch 버전을 바꾸기 때문)
!pip -q install vllm
import os; os.kill(os.getpid(), 9)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, re, time
import pandas as pd
BASE = '/content/drive/MyDrive/항만_llm_aug'
print(os.listdir(BASE))
inp = pd.read_csv(f'{BASE}/llm_aug_input.csv')
fs = pd.read_csv(f'{BASE}/llm_fewshot_sets.csv')
print(len(inp), 'names', fs.groupby('set_id').size().to_dict())

In [ ]:
RULES = {
    0: ('You write the Hangul name that a Korean port authority would record for a ship. '
        'Transliterate by pronunciation (외래어 표기); do not translate meanings. '
        'Acronym letters are read as Korean letter names (MS -> 엠에스, KMTC -> 케이엠티씨). \"No. N\" becomes 제N.'),
    1: ('You are a Korean port clerk typing a ship name heard over the radio into the port record in Hangul. '
        'Write how it sounds to a Korean ear; spaces between words may be kept or dropped, and 호 may be added. '
        'Never translate meanings. Acronyms are spelled with Korean letter names.'),
    2: ('Write the ship name in Hangul the way Korean shipping companies commonly write foreign ship names. '
        'Follow the pronunciation of each word; keep numbers as digits. Do not translate meanings. '
        'Acronyms use Korean letter names (e.g. SJ -> 에스제이).'),
}
def system(pid):
    ex = fs[fs.set_id == pid]
    return RULES[pid] + ' Answer with the Hangul name only.\n\nExamples:\n' + '\n'.join(
        f'English: {e}\nKorean: {k}' for e, k in zip(ex.en_name, ex.ko_name))

def clean(s):
    s = s.strip().split('\n')[0]
    s = re.sub(r'^(Korean|한글)\s*[:：]\s*', '', s)
    s = re.sub(r'[^0-9A-Za-z가-힣 ]', ' ', s)
    return ' '.join(s.split())

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model='Qwen/Qwen2.5-7B-Instruct', dtype='bfloat16', max_model_len=2048,
          enable_prefix_caching=True, gpu_memory_utilization=0.90, seed=0)
tok = llm.get_tokenizer()
t = time.time(); rows = []
for pid in RULES:
    prompts = [tok.apply_chat_template([{'role': 'system', 'content': system(pid)},
                                        {'role': 'user', 'content': f'English: {e}\nKorean:'}],
                                       tokenize=False, add_generation_prompt=True) for e in inp.en_name]
    greedy = llm.generate(prompts, SamplingParams(temperature=0.0, max_tokens=32, stop=['\n']))
    samp = llm.generate(prompts, SamplingParams(n=2, temperature=0.9, top_p=0.95, max_tokens=32, stop=['\n'], seed=100 + pid))
    for (v, e), g, s in zip(inp[['vessel_id', 'en_name']].itertuples(index=False), greedy, samp):
        rows.append((v, e, pid, 'greedy', clean(g.outputs[0].text)))
        rows += [(v, e, pid, f'sample{j}', clean(o.text)) for j, o in enumerate(s.outputs)]
    print(f'prompt {pid} done, {time.time() - t:.0f}s', flush=True)
out = pd.DataFrame(rows, columns=['vessel_id', 'en_name', 'prompt_id', 'kind', 'ko'])
out.to_csv(f'{BASE}/llm_korean_aug_v2.csv', index=False, encoding='utf-8-sig')
uniq = out[out.ko.str.contains('[가-힣]') & ~out.ko.str.contains('[A-Za-z]')].drop_duplicates(['vessel_id', 'ko'])
print(len(out), 'rows |', len(uniq), 'unique Hangul names |', round(len(uniq) / len(inp), 2), 'per vessel')
print(uniq.sample(20, random_state=0)[['en_name', 'prompt_id', 'kind', 'ko']].to_string())

In [ ]:
# 품질 확인: 예시로 쓰지 않은 train 실제 한글 표기와 비교 (지시문별)
from difflib import SequenceMatcher
ref = pd.read_csv(f'{BASE}/train_korean_ref.csv')
m = ref[~ref.vessel_id.isin(fs.vessel_id)].merge(out[out.kind == 'greedy'], on='vessel_id')
m['sim'] = [SequenceMatcher(None, a.replace(' ', ''), str(b).replace(' ', '')).ratio() for a, b in zip(m.ko_name, m.ko)]
m['exact'] = m.ko_name.str.replace(' ', '') == m.ko.astype(str).str.replace(' ', '')
print(m.groupby('prompt_id')[['exact', 'sim']].mean().round(3))